In [ ]:
import cv2
import numpy as np
import math
import requests


# =======================================
# CONFIGURATION
# =======================================
URL = "http://10.29.108.33:8080/shot.jpg"     # <-- CHANGE TO YOUR PHONE IP
WINDOW_WIDTH  = 640                            # Final display width
WINDOW_HEIGHT = 480                            # Final display height
TIMEOUT = 0.3                                   # Fast snapshot timeout

FOCAL_LENGTH = 500.0
LANDMARK_REAL_HEIGHT_CM = 10.0


# =======================================
# HSV COLOR RANGES
# =======================================
lower_red1  = np.array([0, 120, 70])
upper_red1  = np.array([10, 255, 255])
lower_red2  = np.array([170, 120, 70])
upper_red2  = np.array([180, 255, 255])

lower_green = np.array([35, 80, 80])
upper_green = np.array([85, 255, 255])

lower_blue  = np.array([75, 40, 40])
upper_blue  = np.array([145, 255, 255])


# =======================================
# HELPER FUNCTIONS
# =======================================

def find_all_contours(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cnts


def filter_entities(cnts, 
                    min_area=450,
                    max_area=150000,
                    min_w=10,
                    min_h=10,
                    aspect=(0.3, 5.0)):

    out = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        area = w*h

        if area < min_area or area > max_area:
            continue
        if w < min_w or h < min_h:
            continue
        
        ar = w / float(h)
        if not (aspect[0] <= ar <= aspect[1]):
            continue
        
        out.append(c)

    return out


def box_and_center(cnt):
    x, y, w, h = cv2.boundingRect(cnt)
    return (x, y, w, h), (x + w/2, y + h/2)


def match_landmark(top_cnts, bottom_cnts, orientation):
    matches = []

    for top in top_cnts:
        top_box, (tx, ty) = box_and_center(top)

        best = None
        best_d = 1e9
        best_box = None

        for bottom in bottom_cnts:
            b_box, (bx, by) = box_and_center(bottom)
            d = math.hypot(tx - bx, ty - by)

            if d < best_d:
                best_d = d
                best = bottom
                best_box = b_box

        if best is None or best_d > 250:
            continue

        (bx, by, bw, bh) = best_box
        (tx, ty, tw, th) = top_box

        top_bottom = ty + th
        bottom_center = by + bh/2

        if orientation == "TOP_ABOVE":
            if not (top_bottom < bottom_center):
                continue
        else:
            if not (ty > bottom_center):
                continue

        overlap = max(0, min(tx+tw, bx+bw) - max(tx, bx))
        if overlap < 0.25 * min(tw, bw):
            continue

        full_top = min(ty, by)
        full_bottom = max(ty + th, by + bh)
        hpx = full_bottom - full_top
        if hpx <= 0:
            continue

        matches.append((top_box, best_box, hpx))

    return matches


# =======================================
# MAIN LOOP (snapshot mode)
# =======================================
while True:

    # ------- Fetch frame from phone --------
    try:
        resp = requests.get(URL, timeout=TIMEOUT)
        frame = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
        if frame is None:
            print("⚠ Frame decode error")
            continue
    except:
        print("⚠ Connection timeout")
        continue

    # Resize output window
    frame = cv2.resize(frame, (WINDOW_WIDTH, WINDOW_HEIGHT))
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)


    # =======================================
    # MASKS
    # =======================================

    # GREEN
    mask_green = cv2.inRange(hsv, lower_green, upper_green)

    # RED
    mask_red = (
        cv2.inRange(hsv, lower_red1, upper_red1) |
        cv2.inRange(hsv, lower_red2, upper_red2)
    )

    # BLUE (robust)
    hsv_blur = cv2.GaussianBlur(hsv, (7,7), 0)
    mask_blue = cv2.inRange(hsv_blur, lower_blue, upper_blue)
    kernel = np.ones((5,5), np.uint8)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel)
    mask_blue = cv2.medianBlur(mask_blue, 7)

    # Combined mask for display
    mask_combined = mask_green | mask_red | mask_blue


    # =======================================
    # EXTRACT ENTITIES (very stable)
    # =======================================
    green_cnts = filter_entities(find_all_contours(mask_green))
    red_cnts   = filter_entities(find_all_contours(mask_red))
    blue_cnts  = filter_entities(find_all_contours(mask_blue))


    # =======================================
    # 4 LANDMARK DEFINITIONS
    # =======================================
    landmarks = [
        ("RED-GREEN",   red_cnts,   green_cnts, "TOP_ABOVE", (0,255,255)),
        ("BLUE-GREEN",  blue_cnts,  green_cnts, "TOP_ABOVE", (255,255,0)),
        ("GREEN-RED",   green_cnts, red_cnts,   "TOP_ABOVE", (0,255,0)),
        ("GREEN-BLUE",  green_cnts, blue_cnts,  "TOP_ABOVE", (0,128,255)),
    ]


    # =======================================
    # PROCESS LANDMARKS
    # =======================================
    for name, tops, bottoms, orientation, color in landmarks:
        results = match_landmark(tops, bottoms, orientation)

        for (top_box, bot_box, hpx) in results:
            distance = (LANDMARK_REAL_HEIGHT_CM * FOCAL_LENGTH) / hpx

            (tx, ty, tw, th) = top_box
            (bx, by, bw, bh) = bot_box

            TL = (min(tx, bx), min(ty, by))
            BR = (max(tx+tw, bx+bw), max(ty+th, by+bh))

            cv2.rectangle(frame, TL, BR, color, 3)
            cv2.putText(frame, name, (TL[0], TL[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(frame, f"{distance:.1f} cm",
                        (TL[0], BR[1] + 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)


    # =======================================
    # DISPLAY
    # =======================================
    mask_vis = cv2.cvtColor(mask_combined, cv2.COLOR_GRAY2BGR)
    mask_vis = cv2.resize(mask_vis, (WINDOW_WIDTH, WINDOW_HEIGHT))

    combined = np.hstack((frame, mask_vis))
    cv2.imshow("Landmarks (left)  |  Mask (right)", combined)

    # exit with ESC
    if cv2.waitKey(1) == 27:
        break

cv2.destroyAllWindows()


⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection timeout
⚠ Connection 

In [ ]:
import socket
import time

# **1. Configuration - REPLACE THIS IP ADDRESS**
ESP32_IP = "192.168.100.83"  # <--- REPLACE with the actual IP address printed by the ESP32
PORT = 8888  # Must match the port defined in the ESP32 sketch

def send_and_receive(command):
    """Connects to the ESP32, sends a command, and prints the response."""
    # Ensure command ends with a newline character (\n) for the ESP32 to read it completely
    full_command = command + '\n' 
    
    print(f"\n--- Attempting to send: '{command}' ---")
    
    try:
        # Create a socket object (AF_INET for IPv4, SOCK_STREAM for TCP)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            # Connect to the ESP32
            s.connect((ESP32_IP, PORT))
            print("Connection successful. Sending data...")
            
            # Send the command encoded as bytes
            s.sendall(full_command.encode('utf-8'))
            
            # Wait for and receive the response (up to 1024 bytes)
            response = s.recv(1024)
            
            # Decode the response and print it
            print(f"ESP32 Response: {response.decode('utf-8').strip()}")
            
    except ConnectionRefusedError:
        print(f"ERROR: Connection refused. Check if ESP32 is running and IP ({ESP32_IP}) is correct.")
    except TimeoutError:
        print("ERROR: Connection timed out.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# **2. Trial Commands**

# Turn the LED ON
send_and_receive("LED_ON")
time.sleep(1) # Wait for 1 second

# Turn the LED OFF
send_and_receive("LED_OFF")
time.sleep(1)

# Send an unknown command to test error response
send_and_receive("MOVE_FORWARD")

In [ ]:
import cv2
import numpy as np
import math
import requests
import socket
import time

# =======================================
# CONFIGURATION
# =======================================
# Phone camera configuration
PHONE_URL = "http://10.29.108.33:8080/shot.jpg"  # Phone IP for camera
WINDOW_WIDTH  = 640
WINDOW_HEIGHT = 480
TIMEOUT = 0.3

# ESP32 configuration
ESP32_IP = "192.168.100.83"  # ESP32 IP address
ESP32_PORT = 8888

# Landmark parameters
FOCAL_LENGTH = 500.0
LANDMARK_REAL_HEIGHT_CM = 10.0

# =======================================
# HSV COLOR RANGES
# =======================================
lower_red1  = np.array([0, 120, 70])
upper_red1  = np.array([10, 255, 255])
lower_red2  = np.array([170, 120, 70])
upper_red2  = np.array([180, 255, 255])

lower_green = np.array([35, 80, 80])
upper_green = np.array([85, 255, 255])

lower_blue  = np.array([75, 40, 40])
upper_blue  = np.array([145, 255, 255])


# =======================================
# ESP32 COMMUNICATION
# =======================================
def send_to_esp32(command):
    """Send command to ESP32 and return success status."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(2.0)  # 2 second timeout
            s.connect((ESP32_IP, ESP32_PORT))
            s.sendall((command + '\n').encode('utf-8'))
            response = s.recv(1024)
            print(f"✓ ESP32: {response.decode('utf-8').strip()}")
            return True
    except Exception as e:
        print(f"✗ ESP32 Error: {e}")
        return False


# =======================================
# HELPER FUNCTIONS
# =======================================
def find_all_contours(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return cnts


def filter_entities(cnts, 
                    min_area=450,
                    max_area=150000,
                    min_w=10,
                    min_h=10,
                    aspect=(0.3, 5.0)):
    out = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        area = w*h

        if area < min_area or area > max_area:
            continue
        if w < min_w or h < min_h:
            continue
        
        ar = w / float(h)
        if not (aspect[0] <= ar <= aspect[1]):
            continue
        
        out.append(c)
    return out


def box_and_center(cnt):
    x, y, w, h = cv2.boundingRect(cnt)
    return (x, y, w, h), (x + w/2, y + h/2)


def match_landmark(top_cnts, bottom_cnts, orientation):
    matches = []

    for top in top_cnts:
        top_box, (tx, ty) = box_and_center(top)

        best = None
        best_d = 1e9
        best_box = None

        for bottom in bottom_cnts:
            b_box, (bx, by) = box_and_center(bottom)
            d = math.hypot(tx - bx, ty - by)

            if d < best_d:
                best_d = d
                best = bottom
                best_box = b_box

        if best is None or best_d > 250:
            continue

        (bx, by, bw, bh) = best_box
        (tx, ty, tw, th) = top_box

        top_bottom = ty + th
        bottom_center = by + bh/2

        if orientation == "TOP_ABOVE":
            if not (top_bottom < bottom_center):
                continue
        else:
            if not (ty > bottom_center):
                continue

        overlap = max(0, min(tx+tw, bx+bw) - max(tx, bx))
        if overlap < 0.25 * min(tw, bw):
            continue

        full_top = min(ty, by)
        full_bottom = max(ty + th, by + bh)
        hpx = full_bottom - full_top
        if hpx <= 0:
            continue

        matches.append((top_box, best_box, hpx))

    return matches


# =======================================
# MAIN LOOP WITH ESP32 INTEGRATION
# =======================================
landmark_detected_state = {}  # Track detection state for each landmark type

print("Starting landmark detection system...")
print(f"Phone camera: {PHONE_URL}")
print(f"ESP32: {ESP32_IP}:{ESP32_PORT}")
print("Press ESC to exit\n")

while True:
    # ------- Fetch frame from phone --------
    try:
        resp = requests.get(PHONE_URL, timeout=TIMEOUT)
        frame = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
        if frame is None:
            print("⚠ Frame decode error")
            continue
    except:
        print("⚠ Connection timeout")
        continue

    # Resize and convert
    frame = cv2.resize(frame, (WINDOW_WIDTH, WINDOW_HEIGHT))
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # =======================================
    # CREATE MASKS
    # =======================================
    mask_green = cv2.inRange(hsv, lower_green, upper_green)
    
    mask_red = (
        cv2.inRange(hsv, lower_red1, upper_red1) |
        cv2.inRange(hsv, lower_red2, upper_red2)
    )
    
    hsv_blur = cv2.GaussianBlur(hsv, (7,7), 0)
    mask_blue = cv2.inRange(hsv_blur, lower_blue, upper_blue)
    kernel = np.ones((5,5), np.uint8)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel)
    mask_blue = cv2.medianBlur(mask_blue, 7)

    mask_combined = mask_green | mask_red | mask_blue

    # =======================================
    # EXTRACT ENTITIES
    # =======================================
    green_cnts = filter_entities(find_all_contours(mask_green))
    red_cnts   = filter_entities(find_all_contours(mask_red))
    blue_cnts  = filter_entities(find_all_contours(mask_blue))

    # =======================================
    # LANDMARK DEFINITIONS
    # =======================================
    landmarks = [
        ("RED-GREEN",   red_cnts,   green_cnts, "TOP_ABOVE", (0,255,255)),
        ("BLUE-GREEN",  blue_cnts,  green_cnts, "TOP_ABOVE", (255,255,0)),
        ("GREEN-RED",   green_cnts, red_cnts,   "TOP_ABOVE", (0,255,0)),
        ("GREEN-BLUE",  green_cnts, blue_cnts,  "TOP_ABOVE", (0,128,255)),
    ]

    # Track which landmarks are currently detected
    currently_detected = set()

    # =======================================
    # PROCESS LANDMARKS
    # =======================================
    for name, tops, bottoms, orientation, color in landmarks:
        results = match_landmark(tops, bottoms, orientation)

        for (top_box, bot_box, hpx) in results:
            distance = (LANDMARK_REAL_HEIGHT_CM * FOCAL_LENGTH) / hpx

            (tx, ty, tw, th) = top_box
            (bx, by, bw, bh) = bot_box

            TL = (min(tx, bx), min(ty, by))
            BR = (max(tx+tw, bx+bw), max(ty+th, by+bh))

            # Draw on frame
            cv2.rectangle(frame, TL, BR, color, 3)
            cv2.putText(frame, name, (TL[0], TL[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(frame, f"{distance:.1f} cm",
                        (TL[0], BR[1] + 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
            # Mark as detected
            currently_detected.add(name)

    # =======================================
    # ESP32 COMMUNICATION LOGIC
    # =======================================
    # Check for newly detected landmarks
    for landmark_name in currently_detected:
        if landmark_name not in landmark_detected_state or not landmark_detected_state[landmark_name]:
            print(f"\n🎯 DETECTED: {landmark_name}")
            send_to_esp32(f"LANDMARK_DETECTED_{landmark_name.replace('-', '_')}")
            landmark_detected_state[landmark_name] = True

    # Check for landmarks that are no longer detected
    for landmark_name in list(landmark_detected_state.keys()):
        if landmark_detected_state[landmark_name] and landmark_name not in currently_detected:
            print(f"\n❌ LOST: {landmark_name}")
            send_to_esp32(f"LANDMARK_LOST_{landmark_name.replace('-', '_')}")
            landmark_detected_state[landmark_name] = False

    # =======================================
    # DISPLAY
    # =======================================
    mask_vis = cv2.cvtColor(mask_combined, cv2.COLOR_GRAY2BGR)
    mask_vis = cv2.resize(mask_vis, (WINDOW_WIDTH, WINDOW_HEIGHT))

    combined = np.hstack((frame, mask_vis))
    
    # Add status text
    status_text = f"Detected: {len(currently_detected)} landmarks"
    cv2.putText(combined, status_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    cv2.imshow("Landmarks (left)  |  Mask (right)", combined)

    # Exit with ESC
    if cv2.waitKey(1) == 27:
        break

cv2.destroyAllWindows()
print("\nSystem stopped.")